In [1]:
import sys, json; sys.path.append("..")
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from src.train import make_lgbm

train = pd.read_csv("../data/processed/train.csv", index_col=0)
y = train.pop("SeriousDlqin2yrs"); X = train
best = json.load(open("../reports/best_lgbm_params.json"))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

raw = cross_val_predict(make_lgbm(**best), X, y, cv=cv, method="predict_proba")[:, 1]
actual, predicted = calibration_curve(y, raw, n_bins=10)
pd.DataFrame({"model_says": predicted, "actually_defaulted": actual}).round(3)

,model_says,actually_defaulted
0,0.066,0.006
1,0.144,0.012
2,0.246,0.025
3,0.348,0.041
4,0.448,0.060
5,0.549,0.085
6,0.648,0.120
7,0.748,0.189
8,0.852,0.308
9,0.936,0.540


In [2]:
calibrated = CalibratedClassifierCV(make_lgbm(**best), method="isotonic", cv=3)
cal = cross_val_predict(calibrated, X, y, cv=cv, method="predict_proba")[:, 1]
actual, predicted = calibration_curve(y, cal, n_bins=10)
pd.DataFrame({"model_says": predicted, "actually_defaulted": actual}).round(3)

,model_says,actually_defaulted
0,0.025,0.025
1,0.137,0.138
2,0.245,0.239
3,0.347,0.348
4,0.444,0.448
5,0.551,0.569
6,0.643,0.624
7,0.735,0.715
8,0.839,0.759
9,0.937,0.842
